In [1]:
!pip install catboost & lightgbm --config-settings=cmake.define.USE_GPU=ON

/bin/bash: line 1: lightgbm: command not found


In [2]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, RandomizedSearchCV

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.columns)

Index(['id', 'age', 'alcohol_consumption_per_week',
       'physical_activity_minutes_per_week', 'diet_score',
       'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi',
       'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate',
       'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol',
       'triglycerides', 'gender', 'ethnicity', 'education_level',
       'income_level', 'smoking_status', 'employment_status',
       'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history', 'diagnosed_diabetes'],
      dtype='object')


In [4]:
def engineer_features(df):
    # คัดลอก DataFrame เพื่อไม่ให้กระทบตัวแปรเดิม
    df = df.copy()

    # --- 1. กลุ่มดัชนีร่างกาย (Body Metrics) ---
    # สัดส่วนเอวต่อสะโพก (Waist-to-Hip Ratio) บ่งบอกไขมันช่องท้อง
    # (มีมาให้แล้ว แต่เราสามารถแบ่งกลุ่มตามความเสี่ยงได้)
    df['is_obese'] = (df['bmi'] >= 30).astype(int)
    df['high_whr'] = df.apply(lambda x: 1 if (x['gender'] == 'Male' and x['waist_to_hip_ratio'] > 0.9)
                              else (1 if x['waist_to_hip_ratio'] > 0.85 else 0), axis=1)

    # --- 2. กลุ่มความดันโลหิต (Blood Pressure) ---
    # Pulse Pressure: ค่าความต่างของความดันตัวบนและตัวล่าง
    df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
    # Mean Arterial Pressure (MAP): ความดันเฉลี่ยในหลอดเลือด
    df['map'] = (df['systolic_bp'] + 2 * df['diastolic_bp']) / 3
    # Hypertension Stage (เกณฑ์ความดันสูง)
    df['is_hypertensive'] = ((df['systolic_bp'] >= 140) | (df['diastolic_bp'] >= 90)).astype(int)

    # --- 3. กลุ่มไขมันในเลือด (Lipid Profile) ---
    # Total Cholesterol / HDL Ratio (ยิ่งสูงยิ่งเสี่ยง)
    df['chol_hdl_ratio'] = df['cholesterol_total'] / (df['hdl_cholesterol'] + 1e-9)
    # LDL / HDL Ratio
    df['ldl_hdl_ratio'] = df['ldl_cholesterol'] / (df['hdl_cholesterol'] + 1e-9)
    # Triglyceride / HDL Ratio (ตัวบ่งชี้ภาวะดื้ออินซูลินที่แม่นยำมาก)
    df['tg_hdl_ratio'] = df['triglycerides'] / (df['hdl_cholesterol'] + 1e-9)
    # Non-HDL Cholesterol
    df['non_hdl_cholesterol'] = df['cholesterol_total'] - df['hdl_cholesterol']

    # --- 4. กลุ่มไลฟ์สไตล์ (Lifestyle Interactions) ---
    # อัตราส่วนกิจกรรมต่อการจ้องจอ (Active vs Sedentary)
    df['activity_screen_ratio'] = df['physical_activity_minutes_per_week'] / (df['screen_time_hours_per_day'] * 7 + 1)
    # คะแนนสุขภาพรวม (Healthy Score)
    # (ยิ่งกินดี นอนพอ ออกกำลังกายเยอะ คะแนนยิ่งสูง)
    df['lifestyle_score'] = (df['diet_score'] + (df['sleep_hours_per_day'] * 2)) - (df['alcohol_consumption_per_week'] / 7)

    # --- 5. กลุ่มประวัติครอบครัวและโรคประจำตัว (Risk Factors) ---
    # นับรวมความเสี่ยงทางกรรมพันธุ์และโรคเดิม
    df['total_risk_histories'] = df[['family_history_diabetes', 'hypertension_history', 'cardiovascular_history']].sum(axis=1)

    # --- 6. การจัดการข้อมูลที่เป็นกลุ่ม (Encoding) ---
    # แปลงระดับการศึกษาและรายได้เป็นตัวเลข (Ordinal Encoding)
    # สมมติลำดับจากน้อยไปมาก (ปรับตามข้อมูลจริงของคุณ)
    edu_map = {'High School': 1, 'Associate Degree': 2, 'Bachelor': 3, 'Master': 4, 'PhD': 5}
    income_map = {'Low': 1, 'Medium': 2, 'High': 3}

    if 'education_level' in df.columns:
        df['education_level_coded'] = df['education_level'].map(edu_map).fillna(0)
    if 'income_level' in df.columns:
        df['income_level_coded'] = df['income_level'].map(income_map).fillna(0)

    # --- 7. ลบ Column ที่ไม่จำเป็น ---
    # 'id' ไม่ควรนำมาเทรน
    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    return df

In [5]:
train_fe = engineer_features(train)
test_fe = engineer_features(test)

X = train_fe.drop(columns='diagnosed_diabetes')
y = train_fe['diagnosed_diabetes']

In [6]:
cat_features = ['gender', 'ethnicity', 'smoking_status', 'employment_status']

ord_features = ['education_level', 'income_level']

num_features = [col for col in train_fe.columns if col not in cat_features + ord_features + ['diagnosed_diabetes']]

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('ord', OrdinalEncoder(), ord_features)
    ])

X_transformed = preprocessor.fit_transform(X)

In [8]:
xgb_grid = {
    'n_estimators': [500, 1000],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

lgbm_grid = {
    'n_estimators': [500, 1000],
    'num_leaves': [31, 63, 127],
    'learning_rate': [0.01, 0.05, 0.1],
    'feature_fraction': [0.7, 0.8, 0.9]
}

cat_grid = {
    'iterations': [500, 1000],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7]
}

In [9]:
def tune_model(model, param_grid, X_train, y_train):
  search = RandomizedSearchCV(
      model,
      param_distributions=param_grid,
      n_iter=10,
      cv=3,
      scoring='roc_auc',
      random_state=42,
      verbose=2,
      n_jobs=1
  )

  search.fit(X_train, y_train)
  print(f"Best Score: {search.best_score_}")
  return search.best_params_

In [10]:
print("Tuning XGBoost...")
xgb_model = XGBClassifier(tree_method='hist', device='cuda', random_state=42)
best_xgb_params = tune_model(xgb_model, xgb_grid, X_transformed, y)

print("Tuning LightGBM...")
lgbm_model = LGBMClassifier(device='gpu', random_state=42)
best_lgbm_params = tune_model(lgbm_model, lgbm_grid, X_transformed, y)

print("Tuning CatBoost...")
cat_model = CatBoostClassifier(task_type='GPU', verbose=0, random_state=42)
best_cat_params = tune_model(cat_model, cat_grid, X_transformed, y)

Tuning XGBoost...
Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:35:40] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=8, n_estimators=500, subsample=1.0; total time=   6.8s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=8, n_estimators=500, subsample=1.0; total time=   6.1s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=8, n_estimators=500, subsample=1.0; total time=   6.2s
[CV] END colsample_bytree=1.0, learning_rate=0.01, max_depth=4, n_estimators=500, subsample=0.8; total time=   4.3s
[CV] END colsample_bytree=1.0, learning_rate=0.01, max_depth=4, n_estimators=500, subsample=0.8; total time=   4.3s
[CV] END colsample_bytree=1.0, learning_rate=0.01, max_depth=4, n_estimators=500, subsample=0.8; total time=   5.5s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=4, n_estimators=1000, subsample=1.0; total time=   9.7s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=4, n_estimators=1000, subsample=1.0; total time=   7.8s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=4, n_estimat

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[CV] END feature_fraction=0.8, learning_rate=0.01, n_estimators=500, num_leaves=63; total time=  48.2s
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Number of positive: 290871, number of negative: 175796
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3324
[LightGBM] [Info] Number of data points in the train set: 466667, number of used features: 48
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 28 dense feature groups (12.46 MB)

...

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[CV] END feature_fraction=0.8, learning_rate=0.1, n_estimators=500, num_leaves=127; total time=  35.2s
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Number of positive: 290871, number of negative: 175796
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3324
[LightGBM] [Info] Number of data points in the train set: 466667, number of used features: 48
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 28 dense feature groups (12.46 MB)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[CV] END feature_fraction=0.8, learning_rate=0.1, n_estimators=500, num_leaves=127; total time=  35.2s
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Number of positive: 290872, number of negative: 175795
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3322
[LightGBM] [Info] Number of data points in the train set: 466667, number of used features: 48
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 28 dense feature groups (12.46 MB)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[CV] END feature_fraction=0.8, learning_rate=0.1, n_estimators=500, num_leaves=127; total time=  35.8s
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Info] Number of positive: 436307, number of negative: 263693
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3322
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 48
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 28 dense feature groups (18.69 MB)

In [11]:
tuned_base_models = [
    ('xgb', XGBClassifier(**best_xgb_params, device='cuda', tree_method='hist')),
    ('lgb', LGBMClassifier(**best_lgbm_params, device='gpu')),
    ('cat', CatBoostClassifier(**best_cat_params, task_type='GPU', verbose=0))
]

In [20]:
print(f'best_xbg: {best_xgb_params}\nbest_lgm: {best_lgbm_params}\nbest_cat: {best_cat_params}')

best_xbg: {'subsample': 0.8, 'n_estimators': 1000, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
best_lgm: {'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.1, 'feature_fraction': 0.7}
best_cat: {'learning_rate': 0.1, 'l2_leaf_reg': 1, 'iterations': 500, 'depth': 8}


In [12]:
oof_preds = pd.DataFrame()

for name, model in tuned_base_models:
  print(f'Training {name}...')

  pred = cross_val_predict(model, X_transformed, y, cv=5, method='predict_proba', n_jobs=-1)

  oof_preds[name] = pred[:, 1]

meta_model = LogisticRegression()
meta_model.fit(oof_preds, y)

Training xgb...
Training lgb...


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Training cat...


LogisticRegression()

In [15]:
X_test_transformed = preprocessor.transform(test_fe)

test_meta_input = pd.DataFrame()

print("Generating Test predictions from Base Models...")
for name, model in tuned_base_models:
    model.fit(X_transformed, y)

    test_pred = model.predict_proba(X_test_transformed)[:, 1]
    test_meta_input[name] = test_pred

Generating Test predictions from Base Models...
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Info] Number of positive: 436307, number of negative: 263693
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3323
[LightGBM] [Info] Number of data points in the train set: 700000, number of used features: 48
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 28 dense feature groups (18.69 MB) transferred to GPU in 0.027629 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.623296 -> initscore=0.503561
[LightGBM] [Info] Start training from 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7


In [16]:
final_probs = meta_model.predict_proba(test_meta_input)[:, 1]

print(final_probs)

[0.48341285 0.72526068 0.79760661 ... 0.60899768 0.64978349 0.63371998]


In [17]:
submission = pd.DataFrame({
    "id": test['id'],
    "diagnosed_diabetes": final_probs
})

submission.to_csv("submission.csv", index=False)